# Fundamentals 05 - Lineage Memory API

Este notebook **explora la API** de Lineage Memory. No es un caso OTC; es una pieza core para convertir un `RunResult` en memoria compacta y explicable.

Lineage Memory responde tres preguntas:

```text
que paso -> como paso -> por que la respuesta esta soportada
```

Tambien sirve para pasar contexto compacto a una siguiente llamada sin reenviar todo el trace bruto.

In [ ]:
from __future__ import annotations

import agentic_systems as toolkit

PRETTY = False

## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



## Parametros de `RunPolicy`

`RunPolicy` declara c?mo debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | N?mero m?ximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | N?mero m?ximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | L?mite de tokens del modelo cuando el provider lo soporta. | ?til en providers LM; puede quedar `None` en `python-direct`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion autom?tica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | M?ximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: solo representa la seccion `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico  visible")


## 1) Crear un resultado minimo

Usamos una tool simple con `python-direct` porque `fundamentals` explora la API sin depender de credenciales, Athena ni Bedrock.

In [ ]:
@toolkit.tool
def add_numbers(a: float, b: float) -> dict:
    """Suma dos numeros y devuelve evidencia estructurada."""
    value = a + b
    return {
        "ok": True,
        "operation": "add",
        "a": a,
        "b": b,
        "result": value,
        "summary": f"{a} + {b} = {value}",
    }


result = add_numbers.run({"a": 10, "b": 32})
result.final = toolkit.final_answer(result.data, schema=toolkit.output_schema(["operation", "result"]))

toolkit.show({"run_result": result.normalized()})

## 2) Construir `LineageMemory`

`LineageMemory` no ejecuta nada. Solo proyecta el `RunResult` a una memoria compacta, explicable y serializable.

In [ ]:
memory = toolkit.LineageMemory.from_run_result(
    result,
    name="fundamentals.add_numbers",
    question="Problema aritmetico default",
    goal="Explicar que tool se ejecuto, que evidencia produjo y por que la respuesta es valida.",
    tags=["fundamentals", "lineage"],
)

toolkit.show(memory, title="Lineage Memory  explicacion")

toolkit.show(memory.compact(), title="Lineage Memory  payload serializable")

## 3) Explicacion: que, como y por que

`explain()` prepara una vista directa para notebooks, revisiones humanas y reportes de ejecucion.

In [ ]:
explanation = memory.explain()
toolkit.show(explanation)

## 4) Contexto compacto para ahorrar tokens

`to_prompt_context(...)` es la forma corta de llevar memoria a una llamada posterior. En vez de pasar todo `trace(full)`, pasas solo los hechos importantes.

In [ ]:
compact_context = memory.to_prompt_context(max_chars=900)
savings = memory.estimated_context_savings(result.trace("full"), max_chars=900)

print(compact_context)
toolkit.show(savings)

## 5) Metodo directo desde `RunResult`

Para uso diario, `result.lineage(...)` es el atajo recomendado.

In [ ]:
same_memory = result.lineage(
    name="fundamentals.add_numbers.shortcut",
    question="Problema aritmetico default",
    goal="Mostrar el atajo desde RunResult.",
)

toolkit.show(same_memory, title="Lineage Memory  atajo desde RunResult")

toolkit.show({
    "schema_version": same_memory.schema_version,
    "steps": [step.kind for step in same_memory.steps],
    "answer": same_memory.answer,
}, title="Lineage Memory  campos minimos")

## 6) Convivencia con contratos/policies

Lineage Memory complementa `ContractPolicySpec`: el contrato dice que debia pasar; la memoria explica que paso y que evidencia lo soporta.

In [ ]:
spec = toolkit.ContractPolicySpec(
    name="fundamentals.add_numbers.contract",
    contract=toolkit.AgentContract(
        must_call=["add_numbers"],
        tool_expectation=toolkit.expect.exactly("add_numbers"),
        expected_tool_outputs={"add_numbers": {"ok": True, "operation": "add"}},
    ),
    policy=toolkit.RunPolicy(max_turns=2, max_tool_calls=1, trace="compact"),
)

result.validation = result.validate(spec.contract).to_dict()
contract_memory = result.lineage(
    name="fundamentals.add_numbers.with_contract",
    question="Problema aritmetico default",
    goal="Mostrar contrato + lineage juntos.",
)

toolkit.show(spec.describe(), title="ContractPolicySpec")
toolkit.show(contract_memory, title="Lineage Memory  contrato + ejecucion")

## 7) Render humano

`human_result` sigue renderizando la ejecucion. Con `show_lineage=True`, ademas muestra la seccion **Que paso  Lineage Memory** sin cambiar el contrato base de la API. `LineageMemory` sigue siendo la memoria explicable que puedes guardar, resumir o pasar al siguiente prompt.

In [ ]:
toolkit.human_result(
    result,
    title="Fundamentals  RunResult con Lineage Memory",
    expected_tools=spec.contract.tool_expectation,
    pretty=PRETTY,
    show_lineage=True,
    lineage=contract_memory,
)

toolkit.show({"prompt_context": contract_memory.to_prompt_context(max_chars=900)}, title="Contexto compacto para siguiente prompt")

## Lineage del escenario didactico

Mismo caso, pero empaquetado como una sola tool local para enfocar el notebook en `LineageMemory`: explicacion, evidencia y contexto compacto.

In [ ]:
@toolkit.tool
def solve_default_problem() -> dict:
    """Resuelve el problema aritmetico compartido con procedimiento."""
    return user_problem_payload()


default_result = solve_default_problem.run({})
default_result.final = toolkit.final_answer(
    default_result.data,
    schema=toolkit.output_schema(fields=["procedimiento", "resultado_final", "ok"]),
)

default_lineage = default_result.lineage(
    name="fundamentals.default_problem.lineage",
    question=USER_PROMPT,
    goal="Mostrar Lineage Memory sobre el escenario didactico comun a todos los notebooks.",
)

toolkit.human_result(
    default_result,
    title="Human result + Lineage Memory  escenario didactico",
    expected_tools=toolkit.expect.exactly("solve_default_problem"),
    pretty=PRETTY,
    show_lineage=True,
    lineage=default_lineage,
)

toolkit.show(default_lineage.estimated_context_savings(default_result.trace("full"), max_chars=900))


## Lo importante

- `fundamentals` explora la API; no mete dominio OTC.
- `LineageMemory` se construye desde `RunResult`.
- La memoria explica `que`, `como` y `por que`.
- Sirve para reportar resultados y para reducir contexto en llamadas posteriores.
- No reemplaza observabilidad ni tracing externo; es una proyeccion compacta y portable.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {
        "api": "LineageMemory.from_run_result",
        "description": "Convierte un resultado en huella auditable de ejecucion."
    },
    {
        "api": "result.lineage",
        "description": "Atajo directo para derivar lineage desde el resultado."
    },
    {
        "api": "explain",
        "description": "Explica el flujo con lenguaje humano y trazable."
    },
    {
        "api": "human_text",
        "description": "Genera un texto humano breve y estable desde la huella."
    },
    {
        "api": "to_prompt_context",
        "description": "Empaqueta contexto compacto para ahorrar tokens."
    },
    {
        "api": "estimated_context_savings",
        "description": "Mide cuanto contexto conserva o recorta el lineage."
    },
    {
        "api": "shared scenario lineage",
        "description": "Aplica lineage al mismo problema base del tutorial."
    }
]

toolkit.show({'notebook': '05_lineage_memory_api.ipynb', 'api_coverage': api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `LineageMemory / lineage_memory`: Memoria de trazabilidad.
- `LineageStep`: Paso publico dentro del lineage.
- `LINEAGE_SCHEMA_VERSION`: Version publica del schema lineage.
- `TRACE_SCHEMA_VERSION`: Version publica de trace.
- `maybe_show_trace`: Render seguro de trazas opcionales.
- `chain_history_summary`: Resumen publico de chains/history.

- `Chain / ChainStep`: Tipos publicos para resumir cadenas de pasos cuando el flujo crece mas all? de un solo RunResult.

